<a href="https://colab.research.google.com/github/florvela/IA-y-automatizacion-en-seguridad-defensiva/blob/main/codigos-de-ejemplo/03-pipeline-etl-iocs/03-pipeline-etl-iocs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Módulo 03 — Pipeline ETL e Extracción de IOCs

**Escenario:** El SOC recibe logs de dos fuentes heterogéneas: un servidor SSH bajo ataque de fuerza bruta y un servidor web Apache con tráfico sospechoso. Tu trabajo: construir un pipeline ETL que extraiga ambas fuentes, las normalice al estándar ECS, extraiga los IOCs automáticamente, y los deje listos para ingestar en el SIEM.

**Datasets reales:** Logs de LogHub (repositorio público de logs de producción reales)
- OpenSSH: 2000 líneas de logs SSH reales con ataques de fuerza bruta
- Apache: 2000 líneas de logs HTTP reales con errores y accesos

Objetivos:
- Construir un pipeline ETL completo para seguridad
- Normalizar eventos heterogéneos al schema ECS
- Extraer IOCs con regex (IPs, hashes, URLs, dominios)
- Aplicar whitelisting para reducir falsos positivos

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/03-pipelines-e-iocs/images/pipeline.png" width="520"/>

In [3]:
!pip3 install requests ioc-finder -q


# --- Librerías estándar ---
import re           # Expresiones regulares para extraer IPs, hashes, dominios
import json         # Serializar eventos al formato JSON Lines (JSONL)
import csv          # Exportar IOCs en formato tabular para plataformas de threat intel
import ipaddress    # Validar y clasificar IPs (privada, pública, loopback)
import logging      # Trazabilidad del pipeline: qué pasó, cuándo y con qué resultado
import requests     # Descargar logs desde URLs remotas
from datetime import datetime
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

# Configurar logger con timestamp para poder auditar la ejecución del pipeline
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('pipeline-etl')

# URLs de datasets reales — logs de producción de LogHub
# En un entorno real estas serían colas SQS, buckets S3 o endpoints de API
FUENTES = {
    'ssh':    'https://raw.githubusercontent.com/logpai/loghub/master/OpenSSH/OpenSSH_2k.log',
    'apache': 'https://raw.githubusercontent.com/logpai/loghub/master/Apache/Apache_2k.log',
}

print('Setup OK')

Setup OK


## FASE 1 — EXTRACT: Descargando logs reales

En producción esto sería una cola SQS, un bucket S3, o una API del SIEM. Para la demo, descargamos directamente de LogHub — logs reales de servidores en producción.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/03-pipelines-e-iocs/images/extract.png" width="440"/>

In [4]:
def extraer_logs_url(url: str, nombre: str) -> list[str]:
    """Descarga logs desde una URL. Con retry básico y manejo de errores."""
    logger.info(f'Extrayendo {nombre} desde {url}')
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()  # Lanza excepción si el status HTTP es 4xx o 5xx
        # Filtrar líneas vacías para no procesar ruido innecesario
        lineas = [l for l in resp.text.splitlines() if l.strip()]
        logger.info(f'  → {len(lineas)} líneas extraídas de {nombre}')
        return lineas
    except requests.exceptions.RequestException as e:
        # Retornar lista vacía permite que el pipeline continúe con las otras fuentes
        logger.error(f'Error extrayendo {nombre}: {e}')
        return []


# Descargar ambas fuentes en secuencia
# En producción se paralelizaría con ThreadPoolExecutor o tareas asíncronas
logs_raw = {}
for nombre, url in FUENTES.items():
    logs_raw[nombre] = extraer_logs_url(url, nombre)

# Mostrar una muestra de cómo lucen los logs crudos antes de parsear
print()
print('--- Muestra de logs SSH reales ---')
for linea in logs_raw['ssh'][:3]:
    print(f'  {linea}')

print()
print('--- Muestra de logs Apache reales ---')
for linea in logs_raw['apache'][:3]:
    print(f'  {linea}')


--- Muestra de logs SSH reales ---
  Dec 10 06:55:46 LabSZ sshd[24200]: reverse mapping checking getaddrinfo for ns.marryaldkfaczcz.com [173.234.31.186] failed - POSSIBLE BREAK-IN ATTEMPT!
  Dec 10 06:55:46 LabSZ sshd[24200]: Invalid user webmaster from 173.234.31.186
  Dec 10 06:55:46 LabSZ sshd[24200]: input_userauth_request: invalid user webmaster [preauth]

--- Muestra de logs Apache reales ---
  [Sun Dec 04 04:47:44 2005] [notice] workerEnv.init() ok /etc/httpd/conf/workers2.properties
  [Sun Dec 04 04:47:44 2005] [error] mod_jk child workerEnv in error state 6
  [Sun Dec 04 04:51:08 2005] [notice] jk2_init() Found child 6725 in scoreboard slot 10


## FASE 2 — TRANSFORM: Parsing

Cada fuente habla un idioma distinto. El parser convierte texto crudo en estructuras de datos.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/03-pipelines-e-iocs/images/transform.png" width="440"/>

In [5]:
@dataclass
class EventoRaw:
    """Representación intermedia de un evento parseado antes de normalizar.

    Actúa como contrato entre los parsers y el normalizador ECS:
    cada parser produce un EventoRaw, y normalizar_a_ecs consume uno.
    """
    fuente: str           # 'ssh' o 'apache' — identifica el parser de origen
    timestamp_str: str    # Timestamp en texto (aún no parseado a datetime)
    mensaje: str          # Texto del mensaje de log, sin metadatos
    campos: dict = field(default_factory=dict)  # Campos extraídos específicos de la fuente


# --- Parser SSH ---
# Formato de log SSH (syslog):
#   Dec 10 06:55:46 LabSZ sshd[24200]: Failed password for invalid user webmaster from 173.234.31.186 port 38926 ssh2
PATRON_SSH = re.compile(
    r'(?P<mes>\w+)\s+(?P<dia>\d+)\s+(?P<hora>[\d:]+)\s+'
    r'(?P<host>\S+)\s+(?P<proceso>\S+):\s+(?P<mensaje>.+)'
)
# Extrae la IP de origen: "from 173.234.31.186"
PATRON_IP = re.compile(r'from\s+(?P<ip>[\d.]+)')
# Extrae el usuario: "for invalid user webmaster" o "for root"
PATRON_USUARIO = re.compile(r'(?:for|user)\s+(?:invalid user\s+)?(?P<usuario>\S+)\s+from')

def parsear_ssh(linea: str) -> Optional[EventoRaw]:
    m = PATRON_SSH.match(linea)
    if not m:
        return None  # Línea que no cumple el formato (headers, etc.)
    campos = {}
    ip_m = PATRON_IP.search(m.group('mensaje'))
    if ip_m:
        campos['ip_origen'] = ip_m.group('ip')
    usr_m = PATRON_USUARIO.search(m.group('mensaje'))
    if usr_m:
        campos['usuario'] = usr_m.group('usuario')
    # Clasificar resultado del intento de autenticación
    campos['resultado'] = (
        'fallo' if 'Failed' in m.group('mensaje') or 'Invalid' in m.group('mensaje')
        else 'exitoso' if 'Accepted' in m.group('mensaje')
        else 'info'
    )
    return EventoRaw(
        fuente='ssh',
        timestamp_str=f"{m.group('mes')} {m.group('dia')} {m.group('hora')}",
        mensaje=m.group('mensaje'),
        campos=campos
    )


# --- Parser Apache ---
# Formato de log Apache (error log):
#   [Thu Jun 09 06:07:04 2005] [notice] Apache/2.0.54 (Fedora) configured
PATRON_APACHE = re.compile(
    r'\[(?P<timestamp>[^\]]+)\]\s+\[(?P<nivel>\w+)\]\s+(?P<mensaje>.+)'
)

def parsear_apache(linea: str) -> Optional[EventoRaw]:
    m = PATRON_APACHE.match(linea)
    if not m:
        return None
    return EventoRaw(
        fuente='apache',
        timestamp_str=m.group('timestamp'),
        mensaje=m.group('mensaje'),
        campos={'nivel_log': m.group('nivel')}  # notice, warn, error, crit
    )


# Mapa fuente → función parser (facilita añadir nuevas fuentes sin modificar el loop)
PARSERS = {'ssh': parsear_ssh, 'apache': parsear_apache}

# Parsear todas las fuentes y descartar líneas que no cumplen el formato
eventos_raw = defaultdict(list)
for fuente, lineas in logs_raw.items():
    parser = PARSERS[fuente]
    for linea in lineas:
        evento = parser(linea)
        if evento:
            eventos_raw[fuente].append(evento)

# Reportar tasa de parseo: <100% indica líneas con formato inesperado
for fuente, eventos in eventos_raw.items():
    total = len(logs_raw[fuente])
    parsed = len(eventos)
    print(f'{fuente:<10} {parsed}/{total} líneas parseadas ({parsed/total*100:.1f}%)')

print('\nEjemplo evento SSH parseado:')
e = eventos_raw['ssh'][0]
print(f'  timestamp: {e.timestamp_str}')
print(f'  mensaje:   {e.mensaje[:70]}')
print(f'  campos:    {e.campos}')

ssh        2000/2000 líneas parseadas (100.0%)
apache     2000/2000 líneas parseadas (100.0%)

Ejemplo evento SSH parseado:
  timestamp: Dec 10 06:55:46
  mensaje:   reverse mapping checking getaddrinfo for ns.marryaldkfaczcz.com [173.2
  campos:    {'resultado': 'info'}


## FASE 2 — TRANSFORM: Normalización a ECS

[Elastic Common Schema](https://www.elastic.co/guide/en/ecs/current/index.html) es el estándar para normalizar eventos de seguridad. Con todos los logs en el mismo formato, el SIEM puede correlacionar entre fuentes sin trabajo extra.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/03-pipelines-e-iocs/images/normalizar.jpg" width="440"/>

In [6]:
def normalizar_a_ecs(evento: EventoRaw) -> dict:
    """
    Normaliza un EventoRaw al schema ECS.
    https://www.elastic.co/guide/en/ecs/current/ecs-field-reference.html

    ECS define nombres de campo estandarizados para que logs de distintas fuentes
    sean consultables con la misma query en el SIEM. Por ejemplo:
    - source.ip siempre contiene la IP de origen, sin importar si viene de SSH, firewall o proxy.
    - event.outcome siempre es 'success', 'failure' o 'unknown'.
    """
    ecs = {
        '@timestamp': evento.timestamp_str,
        'event': {
            'dataset': evento.fuente,      # 'ssh' o 'apache'
            'kind': 'event',               # 'event' | 'alert' | 'metric' | 'signal'
            'category': ['authentication'] if evento.fuente == 'ssh' else ['web'],
            'outcome': evento.campos.get('resultado', 'unknown'),
        },
        'log': {
            'original': evento.mensaje     # Siempre conservar el log original (inmutabilidad)
        },
        'message': evento.mensaje,
    }

    # Mapeo de campos específicos de cada fuente a campos ECS estándar
    if 'ip_origen' in evento.campos:
        # ECS: source.ip es el campo canónico para la IP de quien inicia la conexión
        ecs['source'] = {'ip': evento.campos['ip_origen']}
    if 'usuario' in evento.campos:
        # ECS: user.name para el nombre de usuario involucrado en el evento
        ecs['user'] = {'name': evento.campos['usuario']}
    if 'nivel_log' in evento.campos:
        # ECS: log.level para el nivel de severidad (notice, warn, error, crit)
        ecs['log']['level'] = evento.campos['nivel_log'].lower()

    return ecs


# Normalizar todos los eventos de todas las fuentes a una lista plana
eventos_ecs = []
for fuente, eventos in eventos_raw.items():
    for evento in eventos:
        eventos_ecs.append(normalizar_a_ecs(evento))

print(f'Total eventos normalizados a ECS: {len(eventos_ecs)}')
print()
print('Ejemplo evento SSH normalizado a ECS:')
# Buscar el primer evento SSH que tenga IP de origen para que el ejemplo sea representativo
ejemplo = next(e for e in eventos_ecs if e['event']['dataset'] == 'ssh' and 'source' in e)
print(json.dumps(ejemplo, indent=2))

Total eventos normalizados a ECS: 4000

Ejemplo evento SSH normalizado a ECS:
{
  "@timestamp": "Dec 10 06:55:46",
  "event": {
    "dataset": "ssh",
    "kind": "event",
    "category": [
      "authentication"
    ],
    "outcome": "fallo"
  },
  "log": {
    "original": "Invalid user webmaster from 173.234.31.186"
  },
  "message": "Invalid user webmaster from 173.234.31.186",
  "source": {
    "ip": "173.234.31.186"
  },
  "user": {
    "name": "webmaster"
  }
}


## FASE 2 — TRANSFORM: Extracción de IOCs

Un IOC (Indicator of Compromise) es cualquier artefacto observable que indica actividad maliciosa. Los extraemos automáticamente con regex de los mensajes de log.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/03-pipelines-e-iocs/images/IOCs.png" width="440"/>

In [7]:
class ExtractorIOCs:
    """Extrae Indicadores de Compromiso (IOCs) de texto usando regex.

    Los IOCs son artefactos observables que pueden indicar actividad maliciosa:
    IPs, dominios, hashes de archivos, URLs y CVEs encontrados en los logs.
    """

    # Whitelist: rangos de IPs privadas según RFC 1918 y RFC 5735
    # Nunca son IOCs porque pertenecen a redes internas o al propio host
    WHITELIST_REDES = [
        ipaddress.ip_network('10.0.0.0/8'),       # Red de clase A privada
        ipaddress.ip_network('172.16.0.0/12'),     # Red de clase B privada
        ipaddress.ip_network('192.168.0.0/16'),    # Red de clase C privada
        ipaddress.ip_network('127.0.0.0/8'),       # Loopback
    ]

    # Cada patrón captura un tipo específico de IOC:
    PATRONES = {
        'ip':          re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b'),
        # MD5: 32 caracteres hexadecimales (usado en malware hashes)
        'hash_md5':    re.compile(r'\b[a-fA-F0-9]{32}\b'),
        # SHA-256: 64 caracteres hexadecimales (más seguro que MD5)
        'hash_sha256': re.compile(r'\b[a-fA-F0-9]{64}\b'),
        # Dominios con TLDs comunes en campañas de malware
        'dominio':     re.compile(
            r'\b(?:[a-zA-Z0-9](?:[a-zA-Z0-9\-]{0,61}[a-zA-Z0-9])?\.)+'
            r'(?:com|net|org|io|ru|cn|cc|biz|info|co)\b'
        ),
        'url':         re.compile(r'https?://[^\s<>"{}|\\^\[\]`]+'),
        # CVEs: identificadores de vulnerabilidades conocidas
        'cve':         re.compile(r'CVE-\d{4}-\d{4,7}', re.IGNORECASE),
    }

    def es_ip_privada(self, ip_str: str) -> bool:
        """Devuelve True si la IP pertenece a un rango privado o de loopback."""
        try:
            ip = ipaddress.ip_address(ip_str)
            return any(ip in red for red in self.WHITELIST_REDES)
        except ValueError:
            return False  # IP malformada, dejar que pase para revisión manual

    def extraer(self, texto: str) -> dict:
        """Extrae todos los IOCs de un texto, filtrando IPs privadas."""
        iocs = defaultdict(set)
        for tipo, patron in self.PATRONES.items():
            matches = patron.findall(texto)
            for match in matches:
                if tipo == 'ip' and self.es_ip_privada(match):
                    continue  # Filtrar IPs privadas — reducen falsos positivos
                iocs[tipo].add(match)
        # Convertir sets a listas para poder serializar a JSON
        return {k: list(v) for k, v in iocs.items()}

    def extraer_de_eventos(self, eventos_ecs: list) -> list[dict]:
        """Extrae IOCs de una lista de eventos ECS y agrupa los resultados."""
        todos_iocs = []
        for evento in eventos_ecs:
            iocs = self.extraer(evento.get('message', ''))
            # Solo registrar eventos que efectivamente contienen IOCs
            if any(iocs.values()):
                todos_iocs.append({
                    'fuente': evento['event']['dataset'],
                    'timestamp': evento['@timestamp'],
                    'iocs': iocs
                })
        return todos_iocs


extractor = ExtractorIOCs()
resultado_iocs = extractor.extraer_de_eventos(eventos_ecs)

# Consolidar todos los IOCs únicos por tipo a través de todas las fuentes
iocs_consolidados = defaultdict(set)
for r in resultado_iocs:
    for tipo, valores in r['iocs'].items():
        iocs_consolidados[tipo].update(valores)

print('--- IOCs extraídos de logs reales ---')
for tipo, valores in iocs_consolidados.items():
    print(f'  {tipo:<15} {len(valores):>4} únicos')
    for v in list(valores)[:3]:
        print(f'    • {v}')

print(f'\nEventos con IOCs: {len(resultado_iocs)} de {len(eventos_ecs)}')

--- IOCs extraídos de logs reales ---
  ip                62 únicos
    • 207.12.15.211
    • 112.95.230.3
    • 5.36.59.76
  dominio            5 únicos
    • customer-187-141-143-180-sta.uninet-ide.com
    • ec2-52-80-34-196.cn-north-1.compute.amazonaws.com.cn
    • 5.36.59.76.dynamic-dsl-ip.omantel.net

Eventos con IOCs: 1771 de 4000


## FASE 2 — TRANSFORM: Alternativa con `ioc-finder`

Mantener regex propias da control total, pero es tedioso y propenso a errores. `ioc-finder` ofrece patrones ya testeados por la comunidad, soporta más tipos de IOC (incluyendo técnicas MITRE ATT&CK y direcciones Bitcoin) y cubre edge cases que las regex manuales suelen perder.

| Tipo | Campo en `ioc-finder` | Equivalente manual |
|------|-----------------------|--------------------|
| IPs IPv4 | `ipv4s` | `ip` |
| Dominios | `domains` | `dominio` |
| URLs | `urls` | `url` |
| MD5 | `md5s` | `hash_md5` |
| SHA-256 | `sha256s` | `hash_sha256` |
| CVEs | `cves` | `cve` |
| Emails | `email_addresses` | ✗ no implementado |
| MITRE ATT&CK | `attack_techniques` | ✗ no implementado |

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/03-pipelines-e-iocs/images/regex-vs-ioc-finder.png" width="480"/>

In [9]:
from ioc_finder import find_iocs

# Tipos relevantes — ioc-finder devuelve ~30 keys, filtramos las útiles para seguridad
TIPOS_RELEVANTES = {'ipv4s', 'domains', 'urls', 'md5s', 'sha256s', 'sha1s', 'cves', 'email_addresses'}

iocs_lib = defaultdict(set)
for evento in eventos_ecs:
    resultado = find_iocs(evento.get('message', ''))
    for tipo, valores in resultado.items():
        if tipo in TIPOS_RELEVANTES and isinstance(valores, list) and valores:
            iocs_lib[tipo].update(valores)

print('--- IOCs encontrados con ioc-finder ---')
for tipo, valores in sorted(iocs_lib.items()):
    print(f'  {tipo:<22} {len(valores):>4} únicos')
    for v in list(valores)[:2]:
        print(f'    • {v}')

# Comparación directa con el extractor manual
MAPEO = {
    'ipv4s':     'ip',
    'domains':   'dominio',
    'urls':      'url',
    'md5s':      'hash_md5',
    'sha256s':   'hash_sha256',
    'cves':      'cve',
}
print()
print(f"{'Tipo':<22} {'ioc-finder':>12}   {'manual (regex)':>14}")
print('-' * 52)
for tipo_lib, tipo_manual in MAPEO.items():
    n_lib    = len(iocs_lib.get(tipo_lib, []))
    n_manual = len(iocs_consolidados.get(tipo_manual, []))
    print(f'{tipo_lib:<22} {n_lib:>12}   {n_manual:>14}')

--- IOCs encontrados con ioc-finder ---
  domains                   7 únicos
    • 191-210-223-172.user.vivozap.com.br
    • 195-154-37-122.rev.poneytelecom.eu
  ipv4s                    62 únicos
    • 207.12.15.211
    • 112.95.230.3

Tipo                     ioc-finder   manual (regex)
----------------------------------------------------
ipv4s                            62               62
domains                           7                5
urls                              0                0
md5s                              0                0
sha256s                           0                0
cves                              0                0


## FASE 2 — TRANSFORM: Enriquecimiento local (sin APIs externas)

Antes de llegar a VirusTotal (que se ve más adelante en el módulo), podemos enriquecer localmente: clasificar IPs, calcular frecuencia, detectar IPs atacando desde múltiples puertos.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/03-pipelines-e-iocs/images/enriquecer.png" width="440"/>

In [10]:
def enriquecer_ip_local(ip_str: str, eventos_ecs: list) -> dict:
    """Enriquecimiento local de una IP sin APIs externas.

    En el módulo 05 veremos cómo complementar esto con VirusTotal
    para obtener reputación global, categorías de amenaza y scores más precisos.
    """
    try:
        ip = ipaddress.ip_address(ip_str)
        # Clasificar el tipo de IP según su rango
        tipo = (
            'privada'   if ip.is_private   else
            'loopback'  if ip.is_loopback  else
            'multicast' if ip.is_multicast else
            'publica'
        )
    except ValueError:
        tipo = 'invalida'

    # Contar cuántas veces aparece esta IP como source.ip en los eventos ECS
    apariciones = sum(
        1 for e in eventos_ecs
        if e.get('source', {}).get('ip') == ip_str
    )

    # Score local basado únicamente en frecuencia (0–50 puntos)
    # La otra mitad del score la aportará VirusTotal en el módulo 05
    score_local = min(apariciones * 5, 50)

    return {
        'ip': ip_str,
        'tipo': tipo,
        'apariciones': apariciones,
        'score_local': score_local,
        'nota': 'Enriquecimiento completo disponible con VirusTotal en módulo 05'
    }


# Obtener las 5 IPs con más apariciones en los logs
# Counter.most_common() devuelve lista de (elemento, conteo) ordenada por frecuencia
ips_frecuentes = Counter(
    e['source']['ip']
    for e in eventos_ecs
    if 'source' in e  # Solo eventos que tienen campo source.ip
).most_common(5)

print('--- Top IPs en logs reales (enriquecimiento local) ---')
for ip, freq in ips_frecuentes:
    info = enriquecer_ip_local(ip, eventos_ecs)
    print(f"  {ip:<20} tipo={info['tipo']:<10} apariciones={info['apariciones']:>4} score_local={info['score_local']}")

--- Top IPs en logs reales (enriquecimiento local) ---
  183.62.140.253       tipo=publica    apariciones= 580 score_local=50
  187.141.143.180      tipo=publica    apariciones= 189 score_local=50
  103.99.0.122         tipo=publica    apariciones= 126 score_local=50
  112.95.230.3         tipo=publica    apariciones=  54 score_local=50
  5.188.10.180         tipo=publica    apariciones=  30 score_local=50


## FASE 3 — LOAD: Exportar resultados

El pipeline termina guardando los datos en formatos que el SIEM y otras herramientas pueden consumir.

In [11]:
def cargar_jsonl(eventos: list, ruta: str):
    """Guarda eventos como JSON Lines (JSONL) — un objeto JSON por línea.

    JSONL es el formato estándar para ingestar logs en SIEMs como Elasticsearch,
    Splunk o OpenSearch. Cada línea es un JSON válido e independiente, lo que
    permite procesar el archivo en streaming sin cargarlo completo en memoria.
    """
    with open(ruta, 'w', encoding='utf-8') as f:
        for evento in eventos:
            f.write(json.dumps(evento, ensure_ascii=False) + '\n')
    logger.info(f'Guardados {len(eventos)} eventos en {ruta}')


def cargar_csv_iocs(iocs_consolidados: dict, ruta: str):
    """Guarda IOCs como CSV para importar en plataformas de threat intelligence.

    El formato CSV (tipo, valor, fuente) es compatible con MISP, OpenCTI
    y la mayoría de plataformas TIP (Threat Intelligence Platform).
    """
    with open(ruta, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['tipo', 'valor', 'fuente'])
        writer.writeheader()
        for tipo, valores in iocs_consolidados.items():
            for valor in valores:
                writer.writerow({'tipo': tipo, 'valor': valor, 'fuente': 'pipeline-etl'})
    logger.info(f'IOCs exportados a {ruta}')


# Solo guardamos los primeros 100 eventos para la demo (el pipeline completo guarda todos)
cargar_jsonl(eventos_ecs[:100], '/tmp/eventos_ecs.jsonl')
cargar_csv_iocs(iocs_consolidados, '/tmp/iocs_extraidos.csv')

# Verificar que los archivos se crearon correctamente
print('Archivos generados:')
for archivo in ['/tmp/eventos_ecs.jsonl', '/tmp/iocs_extraidos.csv']:
    p = Path(archivo)
    print(f'  {p.name:<30} {p.stat().st_size:>8} bytes')

Archivos generados:
  eventos_ecs.jsonl                 34427 bytes
  iocs_extraidos.csv                 2249 bytes


## Pipeline completo — Todo junto

In [12]:
def ejecutar_pipeline(fuentes: dict, directorio_salida: str = '/tmp') -> dict:
    """Ejecuta el pipeline ETL completo sobre las fuentes configuradas.

    El pipeline sigue el patrón ETL estándar para seguridad:
      Extract  → descargar logs crudos desde cada fuente
      Transform → parsear, normalizar a ECS, extraer IOCs
      Load      → guardar eventos normalizados e IOCs para el SIEM
    """
    salida = Path(directorio_salida)
    extractor = ExtractorIOCs()
    todos_eventos_ecs = []
    todos_iocs = defaultdict(set)

    for nombre, url in fuentes.items():
        # ── EXTRACT ──────────────────────────────────────────────
        lineas = extraer_logs_url(url, nombre)
        if not lineas:
            continue  # Fuente no disponible, continuar con las demás

        # ── TRANSFORM — parsear ──────────────────────────────────
        parser = PARSERS.get(nombre)
        if not parser:
            logger.warning(f'No hay parser para fuente: {nombre}')
            continue
        # Parsear y descartar líneas que no cumplen el formato esperado
        eventos_raw_fuente = [parser(l) for l in lineas]
        eventos_raw_fuente = [e for e in eventos_raw_fuente if e]

        # ── TRANSFORM — normalizar a ECS ─────────────────────────
        eventos_ecs_fuente = [normalizar_a_ecs(e) for e in eventos_raw_fuente]
        todos_eventos_ecs.extend(eventos_ecs_fuente)

        # ── TRANSFORM — extraer IOCs ─────────────────────────────
        for evento in eventos_ecs_fuente:
            iocs = extractor.extraer(evento.get('message', ''))
            for tipo, valores in iocs.items():
                todos_iocs[tipo].update(valores)

    # ── LOAD ─────────────────────────────────────────────────────
    # Guardar eventos ECS completos (para ingestar en SIEM como Elasticsearch)
    cargar_jsonl(todos_eventos_ecs, str(salida / 'eventos_normalizados.jsonl'))
    # Guardar IOCs en CSV (para importar en plataformas TIP como MISP o OpenCTI)
    cargar_csv_iocs(todos_iocs, str(salida / 'iocs.csv'))

    # Devolver estadísticas para auditar la ejecución del pipeline
    estadisticas = {
        'total_eventos_procesados': len(todos_eventos_ecs),
        'total_iocs_extraidos': sum(len(v) for v in todos_iocs.values()),
        'iocs_por_tipo': {k: len(v) for k, v in todos_iocs.items()},
        'fuentes_procesadas': list(fuentes.keys()),
    }
    return estadisticas


# Ejecutar el pipeline completo de extremo a extremo
stats = ejecutar_pipeline(FUENTES)

print('=' * 50)
print('       RESUMEN DEL PIPELINE ETL')
print('=' * 50)
print(f"Fuentes procesadas:       {stats['fuentes_procesadas']}")
print(f"Eventos normalizados:     {stats['total_eventos_procesados']}")
print(f"IOCs únicos extraídos:    {stats['total_iocs_extraidos']}")
print()
print('IOCs por tipo:')
for tipo, cantidad in stats['iocs_por_tipo'].items():
    print(f'  {tipo:<15} {cantidad:>4}')
print('=' * 50)
print('\nArchivos disponibles para ingestar en el SIEM:')
print('  /tmp/eventos_normalizados.jsonl')
print('  /tmp/iocs.csv')

       RESUMEN DEL PIPELINE ETL
Fuentes procesadas:       ['ssh', 'apache']
Eventos normalizados:     4000
IOCs únicos extraídos:    67

IOCs por tipo:
  ip                62
  dominio            5

Archivos disponibles para ingestar en el SIEM:
  /tmp/eventos_normalizados.jsonl
  /tmp/iocs.csv


## De notebook a producción

Este notebook cubre el núcleo del pipeline ETL. En un entorno productivo real se agregarían estas capas adicionales:

| Aspecto | Este notebook | `pipeline_produccion.py` |
|---|---|---|
| **Ejecución** | Manual, celda a celda | Loop schedulado cada N minutos |
| **Retry** | Un solo intento por fuente | Exponential backoff (1s → 2s → 4s) |
| **Paralelismo** | Fuentes en serie | `ThreadPoolExecutor` — todas las fuentes simultáneas |
| **Estado** | Descarga todo siempre | Checkpoint: registra la última ejecución exitosa |
| **Entrypoint** | Notebook | `python pipeline_produccion.py` |

El archivo `pipeline_produccion.py` (disponible junto a este notebook) contiene el mismo pipeline con esas cuatro mejoras incorporadas y está listo para ejecutarse desde la terminal.